# Advanced Retrieval with LangChain

In the following notebook, we'll explore various methods of advanced retrieval using LangChain!

We'll touch on:

- Naive Retrieval
- Best-Matching 25 (BM25)
- Multi-Query Retrieval
- Parent-Document Retrieval
- Contextual Compression (a.k.a. Rerank)
- Ensemble Retrieval
- Semantic chunking

We'll also discuss how these methods impact performance on our set of documents with a simple RAG chain.

There will be two breakout rooms:

- 🤝 Breakout Room Part #1
  - Task 1: Getting Dependencies!
  - Task 2: Data Collection and Preparation
  - Task 3: Setting Up QDrant!
  - Task 4-10: Retrieval Strategies
- 🤝 Breakout Room Part #2
  - Activity: Evaluate with Ragas

# 🤝 Breakout Room Part #1

In [ ]:
import os
import langsmith

# Set up LangSmith for tracking and evaluation
os.environ["LANGSMITH_API_KEY"] = getpass.getpass("Enter your LangSmith API Key:")
os.environ["LANGSMITH_TRACING_V2"] = "true"
os.environ["LANGSMITH_PROJECT"] = "Advanced_Retrieval_Session09"

# Initialize LangSmith client
from langsmith import Client
langsmith_client = Client()

print("LangSmith setup complete!")


## Task 1: Getting Dependencies!

We're going to need a few specific LangChain community packages, like OpenAI (for our [LLM](https://platform.openai.com/docs/models) and [Embedding Model](https://platform.openai.com/docs/guides/embeddings)) and Cohere (for our [Reranker](https://cohere.com/rerank)).

We'll also provide our OpenAI key, as well as our Cohere API key.

In [1]:
import os
import getpass

os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API Key:")

Enter your OpenAI API Key: ········


In [2]:
os.environ["COHERE_API_KEY"] = getpass.getpass("Cohere API Key:")

Cohere API Key: ········


## Task 2: Data Collection and Preparation

We'll be using our Use Case Data once again - this time the strutured data available through the CSV!

### Data Preparation

We want to make sure all our documents have the relevant metadata for the various retrieval strategies we're going to be applying today.

In [3]:
from langchain_community.document_loaders.csv_loader import CSVLoader
from datetime import datetime, timedelta

loader = CSVLoader(
    file_path=f"./data/Projects_with_Domains.csv",
    metadata_columns=[
      "Project Title",
      "Project Domain",
      "Secondary Domain",
      "Description",
      "Judge Comments",
      "Score",
      "Project Name",
      "Judge Score"
    ]
)

synthetic_usecase_data = loader.load()

for doc in synthetic_usecase_data:
    doc.page_content = doc.metadata["Description"]

Let's look at an example document to see if everything worked as expected!

In [4]:
synthetic_usecase_data[0]

Document(metadata={'source': './data/Projects_with_Domains.csv', 'row': 0, 'Project Title': 'InsightAI 1', 'Project Domain': 'Security', 'Secondary Domain': 'Finance / FinTech', 'Description': 'A low-latency inference system for multimodal agents in autonomous systems.', 'Judge Comments': 'Technically ambitious and well-executed.', 'Score': '85', 'Project Name': 'Project Aurora', 'Judge Score': '9.5'}, page_content='A low-latency inference system for multimodal agents in autonomous systems.')

## Task 3: Setting up QDrant!

Now that we have our documents, let's create a QDrant VectorStore with the collection name "Synthetic_Usecases".

We'll leverage OpenAI's [`text-embedding-3-small`](https://openai.com/blog/new-embedding-models-and-api-updates) because it's a very powerful (and low-cost) embedding model.

> NOTE: We'll be creating additional vectorstores where necessary, but this pattern is still extremely useful.

In [5]:
from langchain_community.vectorstores import Qdrant
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = Qdrant.from_documents(
    synthetic_usecase_data,
    embeddings,
    location=":memory:",
    collection_name="Synthetic_Usecases"
)

## Task 4: Naive RAG Chain

Since we're focusing on the "R" in RAG today - we'll create our Retriever first.

### R - Retrieval

This naive retriever will simply look at each review as a document, and use cosine-similarity to fetch the 10 most relevant documents.

> NOTE: We're choosing `10` as our `k` here to provide enough documents for our reranking process later

In [6]:
naive_retriever = vectorstore.as_retriever(search_kwargs={"k" : 10})

### A - Augmented

We're going to go with a standard prompt for our simple RAG chain today! Nothing fancy here, we want this to mostly be about the Retrieval process.

In [10]:
from langchain_core.prompts import ChatPromptTemplate

RAG_TEMPLATE = """\
You are a helpful and kind assistant. Use the context provided below to answer the question.

If you do not know the answer, or are unsure, say you don't know.

Query:
{question}

Context:
{context}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_TEMPLATE)

### G - Generation

We're going to leverage `gpt-4.1-nano` as our LLM today, as - again - we want this to largely be about the Retrieval process.

In [11]:
from langchain_openai import ChatOpenAI

chat_model = ChatOpenAI(model="gpt-4.1-nano")

### LCEL RAG Chain

We're going to use LCEL to construct our chain.

> NOTE: This chain will be exactly the same across the various examples with the exception of our Retriever!

In [12]:
from langchain_core.runnables import RunnablePassthrough
from operator import itemgetter
from langchain_core.output_parsers import StrOutputParser

naive_retrieval_chain = (
    # INVOKE CHAIN WITH: {"question" : "<<SOME USER QUESTION>>"}
    # "question" : populated by getting the value of the "question" key
    # "context"  : populated by getting the value of the "question" key and chaining it into the base_retriever
    {"context": itemgetter("question") | naive_retriever, "question": itemgetter("question")}
    # "context"  : is assigned to a RunnablePassthrough object (will not be called or considered in the next step)
    #              by getting the value of the "context" key from the previous step
    | RunnablePassthrough.assign(context=itemgetter("context"))
    # "response" : the "context" and "question" values are used to format our prompt object and then piped
    #              into the LLM and stored in a key called "response"
    # "context"  : populated by getting the value of the "context" key from the previous step
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's see how this simple chain does on a few different prompts.

> NOTE: You might think that we've cherry picked prompts that showcase the individual skill of each of the retrieval strategies - you'd be correct!

In [13]:
naive_retrieval_chain.invoke({"question" : "What is the most common project domain?"})["response"].content

'Based on the provided data, the most common project domain appears to be "Healthcare / MedTech," which is mentioned multiple times in the dataset.'

In [14]:
naive_retrieval_chain.invoke({"question" : "Were there any usecases about security?"})["response"].content

'Yes, there are use cases related to security. Specifically, the project titled "Pathfinder 24" in the Healthcare / MedTech domain with a secondary focus on Security involves an AI-powered platform that optimizes logistics routes for sustainability.'

In [15]:
naive_retrieval_chain.invoke({"question" : "What did judges have to say about the fintech projects?"})["response"].content

'The judges had generally favorable comments about the fintech projects, highlighting their technical quality, impact, and robustness. For example, one project was described as "A clever solution with measurable environmental benefit," and another as "Solid work with impressive real-world impact." Overall, the judges appreciated the technical ambition, quality of implementation, and potential influence of these projects.'

Overall, this is not bad! Let's see if we can make it better!

## Task 5: Best-Matching 25 (BM25) Retriever

Taking a step back in time - [BM25](https://www.nowpublishers.com/article/Details/INR-019) is based on [Bag-Of-Words](https://en.wikipedia.org/wiki/Bag-of-words_model) which is a sparse representation of text.

In essence, it's a way to compare how similar two pieces of text are based on the words they both contain.

This retriever is very straightforward to set-up! Let's see it happen down below!


In [ ]:
from langchain_community.retrievers import BM25Retriever

bm25_retriever = BM25Retriever.from_documents(synthetic_usecase_data)

We'll construct the same chain - only changing the retriever.

In [21]:
bm25_retrieval_chain = (
    {"context": itemgetter("question") | bm25_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's look at the responses!

In [27]:
bm25_retrieval_chain.invoke({"question" : "What is the most common project domain?"})["response"].content

'Based on the provided data, the most common project domain is not explicitly stated, but from the sample, the Domains mentioned are Productivity Assistants, Legal / Compliance, Data / Analytics, and Healthcare / MedTech. Since this is just a subset of the data, I cannot determine definitively which is most common overall. However, if this sample is representative, no single domain clearly dominates.\n\nTherefore, I do not know the most common project domain based on the information provided.'

In [28]:
bm25_retrieval_chain.invoke({"question" : "Were there any usecases about security?"})["response"].content

'Based on the provided context, there do not appear to be any specific use cases related to security.'

In [29]:
bm25_retrieval_chain.invoke({"question" : "What did judges have to say about the fintech projects?"})["response"].content

'The judges described the fintech-related project "PulseAI 50" as "technically ambitious and well-executed."'

It's not clear that this is better or worse, if only we had a way to test this (SPOILERS: We do, the second half of the notebook will cover this)

#### ❓ Question #1:

Give an example query where BM25 is better than embeddings and justify your answer.

##### ✅ Answer
 1. When exact term matching is needed e.g.  supervised, unsupervise machine learning have specific meaning, that may be missed using semantic retrieval
 2. Finding frequency count of a term. When this is required semantic retrieval will distort frequency count.


## Task 6: Contextual Compression (Using Reranking)

Contextual Compression is a fairly straightforward idea: We want to "compress" our retrieved context into just the most useful bits.

There are a few ways we can achieve this - but we're going to look at a specific example called reranking.

The basic idea here is this:

- We retrieve lots of documents that are very likely related to our query vector
- We "compress" those documents into a smaller set of *more* related documents using a reranking algorithm.

We'll be leveraging Cohere's Rerank model for our reranker today!

All we need to do is the following:

- Create a basic retriever
- Create a compressor (reranker, in this case)

That's it!

Let's see it in the code below!

In [18]:
from langchain.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_cohere import CohereRerank

compressor = CohereRerank(model="rerank-v3.5")
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=naive_retriever
)

Let's create our chain again, and see how this does!

In [19]:
contextual_compression_retrieval_chain = (
    {"context": itemgetter("question") | compression_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

In [20]:
contextual_compression_retrieval_chain.invoke({"question" : "What is the most common project domain?"})["response"].content

'The most common project domain, based on the provided data, appears to be "Synthetic Usecases," as multiple entries mention projects related to synthetic data generation. The specific domains listed include Security, Productivity Assistants, and Healthcare / MedTech. Since the data sample is limited, and "Synthetic Usecases" seems to be the collection, the dominant project domains within that context are related to synthetic data applications across various fields. \n\nIf you are asking about the most common domain among all projects in the dataset, I do not have enough information to determine which one is most frequent overall.'

In [21]:
contextual_compression_retrieval_chain.invoke({"question" : "Were there any usecases about security?"})["response"].content

'Based on the provided context, there are no explicit use cases related to security. The projects mentioned focus on federated learning and privacy in healthcare applications, but none specifically mention security as a primary focus.'

In [22]:
contextual_compression_retrieval_chain.invoke({"question" : "What did judges have to say about the fintech projects?"})["response"].content

'Judges had the following comments about the fintech projects:\n\n- For "Pathfinder 27" in the Finance / FinTech domain, judges complimented it for "excellent code quality and use of open-source libraries."\n- For "PlanPilot 35" in the Finance / FinTech domain, judges described it as "a clever solution with measurable environmental benefit."'

We'll need to rely on something like Ragas to help us get a better sense of how this is performing overall - but it "feels" better!

## Task 7: Multi-Query Retriever

Typically in RAG we have a single query - the one provided by the user.

What if we had....more than one query!

In essence, a Multi-Query Retriever works by:

1. Taking the original user query and creating `n` number of new user queries using an LLM.
2. Retrieving documents for each query.
3. Using all unique retrieved documents as context

So, how is it to set-up? Not bad! Let's see it down below!



In [23]:
from langchain.retrievers.multi_query import MultiQueryRetriever

multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=naive_retriever, llm=chat_model
) 

In [24]:
multi_query_retrieval_chain = (
    {"context": itemgetter("question") | multi_query_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

In [25]:
multi_query_retrieval_chain.invoke({"question" : "What is the most common project domain?"})["response"].content

'The most common project domain among the samples provided is "Customer Support / Helpdesk," which appears multiple times in the dataset.'

In [26]:
multi_query_retrieval_chain.invoke({"question" : "Were there any usecases about security?"})["response"].content

'Yes, there are usecases related to security. For example, one project titled "Pathfinder 24" focuses on an AI-powered platform optimizing logistics routes for sustainability, which is linked to security considerations in the context of healthcare/MedTech. Additionally, another project "SecureNest 49" involves a document summarization and retrieval system for enterprise knowledge bases, which pertains to security and compliance concerns.'

In [27]:
multi_query_retrieval_chain.invoke({"question" : "What did judges have to say about the fintech projects?"})["response"].content

'Judges generally appreciated the fintech projects, often highlighting their strong impact, quality, and potential. For example, one project was described as "Solid work with impressive real-world impact," another as "Well-structured and scalable; good potential for commercialization," and others noted as "Outstanding collaboration and well-articulated research scope," or "Promising idea with robust experimental validation." Overall, the judges\' comments reflect a positive view of the fintech projects, emphasizing their conceptual strength, technical quality, and practical significance.'

#### ❓ Question #2:

Explain how generating multiple reformulations of a user query can improve recall.

##### ✅ Answer

 1. Multi-query allows slightly different retrieval around similar semantic meaning. Different phrasing of same concept wiii be retrieved.
 2. Recall is the percentage of possible relevant answers being recalled.
 3. More retrieval leads to more likelihood of finding the true/relevant answers.


## Task 8: Parent Document Retriever

A "small-to-big" strategy - the Parent Document Retriever works based on a simple strategy:

1. Each un-split "document" will be designated as a "parent document" (You could use larger chunks of document as well, but our data format allows us to consider the overall document as the parent chunk)
2. Store those "parent documents" in a memory store (not a VectorStore)
3. We will chunk each of those documents into smaller documents, and associate them with their respective parents, and store those in a VectorStore. We'll call those "child chunks".
4. When we query our Retriever, we will do a similarity search comparing our query vector to the "child chunks".
5. Instead of returning the "child chunks", we'll return their associated "parent chunks".

Okay, maybe that was a few steps - but the basic idea is this:

- Search for small documents
- Return big documents

The intuition is that we're likely to find the most relevant information by limiting the amount of semantic information that is encoded in each embedding vector - but we're likely to miss relevant surrounding context if we only use that information.

Let's start by creating our "parent documents" and defining a `RecursiveCharacterTextSplitter`.

In [29]:
from langchain.retrievers import ParentDocumentRetriever
from langchain.storage import InMemoryStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from qdrant_client import QdrantClient, models

parent_docs = synthetic_usecase_data
child_splitter = RecursiveCharacterTextSplitter(chunk_size=750)

We'll need to set up a new QDrant vectorstore - and we'll use another useful pattern to do so!

> NOTE: We are manually defining our embedding dimension, you'll need to change this if you're using a different embedding model.

In [30]:
from langchain_qdrant import QdrantVectorStore

client = QdrantClient(location=":memory:")

client.create_collection(
    collection_name="full_documents",
    vectors_config=models.VectorParams(size=1536, distance=models.Distance.COSINE)
)

parent_document_vectorstore = QdrantVectorStore(
    collection_name="full_documents", embedding=OpenAIEmbeddings(model="text-embedding-3-small"), client=client
)

Now we can create our `InMemoryStore` that will hold our "parent documents" - and build our retriever!

In [32]:
store = InMemoryStore()

parent_document_retriever = ParentDocumentRetriever(
    vectorstore = parent_document_vectorstore,
    docstore=store,
    child_splitter=child_splitter,
)

By default, this is empty as we haven't added any documents - let's add some now!

In [33]:
parent_document_retriever.add_documents(parent_docs, ids=None)

We'll create the same chain we did before - but substitute our new `parent_document_retriever`.

In [34]:
parent_document_retrieval_chain = (
    {"context": itemgetter("question") | parent_document_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's give it a whirl!

In [35]:
parent_document_retrieval_chain.invoke({"question" : "What is the most common project domain?"})["response"].content

'Based on the provided data, the most common project domain appears to be "Healthcare / MedTech," as it is mentioned multiple times among the projects listed.'

In [46]:
parent_document_retrieval_chain.invoke({"question" : "Were there any usecases about security?"})["response"].content

'Based on the provided context, there do not appear to be any specific usecases explicitly related to security. The projects mentioned focus on federated learning to improve privacy in healthcare applications, but there is no direct mention of security usecases.'

In [47]:
parent_document_retrieval_chain.invoke({"question" : "What did judges have to say about the fintech projects?"})["response"].content

'Based on the provided context, the judges had the following comments about the fintech projects:\n\n- For the project "SkyForge" in the finance/fintech domain, the judges described it as "A clever solution with measurable environmental benefit."\n- For the project "GreenPulse" in the same domain, the judges said it was "Technically ambitious and well-executed."\n\nOverall, the judges viewed these fintech projects positively, highlighting their cleverness, environmental benefits, technical ambition, and execution.'

Overall, the performance *seems* largely the same. We can leverage a tool like [Ragas]() to more effectively answer the question about the performance.

## Task 9: Ensemble Retriever

In brief, an Ensemble Retriever simply takes 2, or more, retrievers and combines their retrieved documents based on a rank-fusion algorithm.

In this case - we're using the [Reciprocal Rank Fusion](https://plg.uwaterloo.ca/~gvcormac/cormacksigir09-rrf.pdf) algorithm.

Setting it up is as easy as providing a list of our desired retrievers - and the weights for each retriever.

In [48]:
from langchain.retrievers import EnsembleRetriever

retriever_list = [bm25_retriever, naive_retriever, parent_document_retriever, compression_retriever, multi_query_retriever]
equal_weighting = [1/len(retriever_list)] * len(retriever_list)

ensemble_retriever = EnsembleRetriever(
    retrievers=retriever_list, weights=equal_weighting
)

We'll pack *all* of these retrievers together in an ensemble.

In [49]:
ensemble_retrieval_chain = (
    {"context": itemgetter("question") | ensemble_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's look at our results!

In [50]:
ensemble_retrieval_chain.invoke({"question" : "What is the most common project domain?"})["response"].content

'Based on the provided data, the most common project domain appears to be "Healthcare / MedTech," as it is mentioned multiple times in the examples. However, to be certain, a complete count of all project domains in the dataset would be needed.  \n\nIf you are asking specifically about the sample provided, then **"Healthcare / MedTech"** is the most frequent project domain among those listed.'

In [51]:
ensemble_retrieval_chain.invoke({"question" : "Were there any usecases about security?"})["response"].content

'Yes, there was a use case related to security. Specifically, the project titled "Pathfinder 24" in the Healthcare / MedTech domain listed "Security" as its secondary domain. Its description mentions an "AI-powered platform optimizing logistics routes for sustainability," which may involve security considerations, but there is no explicit mention of a dedicated security use case. \n\nAdditionally, another project titled "SecureNest 49" in the E‑commerce / Marketplaces domain, with "Legal / Compliance" as a secondary domain, could imply security and compliance aspects related to enterprise knowledge bases, but again, there is no explicit focus solely on security.\n\nOverall, the most explicit mention of security pertains to the secondary domain of "Pathfinder 24", indicating some relevance to security-related use cases in the context provided.'

In [52]:
ensemble_retrieval_chain.invoke({"question" : "What did judges have to say about the fintech projects?"})["response"].content

'The judges\' comments on the fintech projects were generally positive. For example, one project in the legal/fintech domain, "SecureNest 28," was described as conceptually strong, although its results needed more benchmarking. Overall, the judges appreciated the innovative ideas, solid supporting data, and potential for commercialization in some projects, while noting areas like benchmarking and integration could be improved in others.'

## Task 10: Semantic Chunking

While this is not a retrieval method - it *is* an effective way of increasing retrieval performance on corpora that have clean semantic breaks in them.

Essentially, Semantic Chunking is implemented by:

1. Embedding all sentences in the corpus.
2. Combining or splitting sequences of sentences based on their semantic similarity based on a number of [possible thresholding methods](https://python.langchain.com/docs/how_to/semantic-chunker/):
  - `percentile`
  - `standard_deviation`
  - `interquartile`
  - `gradient`
3. Each sequence of related sentences is kept as a document!

Let's see how to implement this!

We'll use the `percentile` thresholding method for this example which will:

Calculate all distances between sentences, and then break apart sequences of setences that exceed a given percentile among all distances.

In [36]:
from langchain_experimental.text_splitter import SemanticChunker

semantic_chunker = SemanticChunker(
    embeddings,
    breakpoint_threshold_type="percentile"
)

Now we can split our documents.

In [37]:
semantic_documents = semantic_chunker.split_documents(synthetic_usecase_data[:20])

Let's create a new vector store.

In [38]:
semantic_vectorstore = Qdrant.from_documents(
    semantic_documents,
    embeddings,
    location=":memory:",
    collection_name="Synthetic_Usecase_Data_Semantic_Chunks"
)

We'll use naive retrieval for this example.

In [39]:
semantic_retriever = semantic_vectorstore.as_retriever(search_kwargs={"k" : 10})

Finally we can create our classic chain!

In [40]:
semantic_retrieval_chain = (
    {"context": itemgetter("question") | semantic_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

And view the results!

In [41]:
semantic_retrieval_chain.invoke({"question" : "What is the most common project domain?"})["response"].content

'Based on the provided data, the most common project domain is "Legal / Compliance" and "Developer Tools / DevEx," each appearing twice among the sample projects. However, since this is just a small subset of the data, I cannot determine which domain is most common overall with certainty. \n\nIf I consider the entire dataset, I do not have enough information to definitively identify the single most common project domain.'

In [59]:
semantic_retrieval_chain.invoke({"question" : "Were there any usecases about security?"})["response"].content

'Yes, there are use cases related to security. Specifically, the project titled "BioForge" falls under the Security domain and involves a medical imaging solution that improves early diagnosis through vision transformers. Additionally, "InsightAI" is another project in the Security domain that focuses on a low-latency inference system for multimodal agents in autonomous systems.'

In [60]:
semantic_retrieval_chain.invoke({"question" : "What did judges have to say about the fintech projects?"})["response"].content

'Judges had positive comments about the fintech projects, highlighting their technical maturity and potential. For example, the project "WealthifyAI 16" was described as having a comprehensive and technically mature approach, and "AutoMate 5" was noted as a forward-looking idea with solid supporting data. Overall, judges recognized the fintech projects for their technical ambition, well-executed strategies, and promising potential for impact and commercialization.'

#### ❓ Question #3:

If sentences are short and highly repetitive (e.g., FAQs), how might semantic chunking behave, and how would you adjust the algorithm?

##### ✅ Answer

 1. For FAQs specifically, there is predictable structure in (Q: ... A: ...)
 2. Clear boundaries between questions, answers
 3. Semantic similarity is misleading (all Q&As are "related" to the topic)

In short clear partitioning , regex methods would be idea. In langchain this is implemented wtih RecursiveCharacterTextSplitter. It's designed for exactly this scenario - structured, repetitive content where semantic boundaries don't align with logical content boundaries.


# 🤝 Breakout Room Part #2

#### 🏗️ Activity #1

Your task is to evaluate the various Retriever methods against eachother.

You are expected to:

1. Create a "golden dataset"
 - Use Synthetic Data Generation (powered by Ragas, or otherwise) to create this dataset
2. Evaluate each retriever with *retriever specific* Ragas metrics
 - Semantic Chunking is not considered a retriever method and will not be required for marks, but you may find it useful to do a "semantic chunking on" vs. "semantic chunking off" comparision between them
3. Compile these in a list and write a small paragraph about which is best for this particular data and why.

Your analysis should factor in:
  - Cost
  - Latency
  - Performance

> NOTE: This is **NOT** required to be completed in class. Please spend time in your breakout rooms creating a plan before moving on to writing code.

##### HINTS:

- LangSmith provides detailed information about latency and cost.

In [ ]:
# Import RAGAS components for evaluation
from ragas.testset import TestsetGenerator
from ragas.testset.evolutions import simple, reasoning, multi_context
from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy, 
    context_recall,
    context_precision
)
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
import pandas as pd

# Wrap our existing LLM and embeddings for RAGAS
ragas_llm = LangchainLLMWrapper(chat_model)
ragas_embeddings = LangchainEmbeddingsWrapper(embeddings)

print("RAGAS components setup complete!")

In [ ]:
# Generate golden dataset using RAGAS TestsetGenerator
print("Generating golden dataset with RAGAS...")

# Create testset generator
generator = TestsetGenerator(
    llm=ragas_llm,
    embeddings=ragas_embeddings
)

# Generate test dataset with 15 examples
# Using the synthetic_usecase_data that was already loaded from CSV
golden_dataset = generator.generate_with_langchain_docs(
    synthetic_usecase_data, 
    testset_size=15,
    distributions={
        simple: 0.5,
        reasoning: 0.3,
        multi_context: 0.2
    }
)

print(f"Generated {len(golden_dataset)} test examples")
print("\nSample from generated dataset:")
print(golden_dataset.head())

# Save the dataset
golden_dataset.to_csv("golden_dataset.csv", index=False)
print("\nGolden dataset saved as 'golden_dataset.csv'")


In [ ]:
# Create LangSmith dataset for evaluation tracking
print("Creating LangSmith dataset...")

try:
    # Create dataset
    dataset_name = "Advanced_Retrieval_Evaluation"
    dataset = langsmith_client.create_dataset(
        dataset_name=dataset_name,
        description="Golden dataset for evaluating different retriever methods"
    )
    print(f"Created dataset: {dataset_name}")
    
    # Upload examples to LangSmith dataset
    examples = []
    for idx, row in golden_dataset.iterrows():
        example = {
            "inputs": {"question": row["question"]},
            "outputs": {"answer": row["answer"]},
            "metadata": {
                "reference_contexts": row["contexts"],
                "ground_truth": row["answer"]
            }
        }
        examples.append(example)
    
    # Upload examples in batches
    langsmith_client.create_examples(examples, dataset_id=dataset.id)
    print(f"Uploaded {len(examples)} examples to LangSmith dataset")
    
except Exception as e:
    print(f"Dataset might already exist or error occurred: {e}")
    # Try to get existing dataset
    try:
        dataset = langsmith_client.read_dataset(dataset_name=dataset_name)
        print(f"Using existing dataset: {dataset_name}")
    except:
        print("Could not create or find dataset. Continuing with evaluation...")


In [ ]:
# Helper function to evaluate a retriever chain
def evaluate_retriever_chain(chain, retriever_name, golden_dataset):
    """
    Evaluate a retriever chain using the golden dataset
    Returns results in RAGAS-compatible format
    """
    print(f"Evaluating {retriever_name} retriever...")
    
    results = []
    
    for idx, row in golden_dataset.iterrows():
        try:
            # Run the chain with LangSmith tracking
            response = chain.invoke(
                {"question": row["question"]},
                config={"metadata": {"retriever": retriever_name}}
            )
            
            # Extract answer and context
            answer = response["response"].content if hasattr(response["response"], 'content') else str(response["response"])
            contexts = [doc.page_content for doc in response["context"]]
            
            # Store result in RAGAS format
            result = {
                "question": row["question"],
                "answer": answer,
                "contexts": contexts,
                "ground_truth": row["answer"],
                "reference_contexts": row["contexts"]
            }
            results.append(result)
            
        except Exception as e:
            print(f"Error evaluating question {idx}: {e}")
            continue
    
    print(f"Completed evaluation for {retriever_name}: {len(results)} successful evaluations")
    return results

print("Evaluation helper function defined!")


In [ ]:
# Create simple ensemble retriever combining naive + BM25
from langchain.retrievers import EnsembleRetriever

# Create simple ensemble with equal weights
simple_ensemble_retriever = EnsembleRetriever(
    retrievers=[naive_retriever, bm25_retriever], 
    weights=[0.5, 0.5]
)

# Create ensemble retrieval chain
simple_ensemble_retrieval_chain = (
    {"context": itemgetter("question") | simple_ensemble_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

print("Simple ensemble retriever created (naive + BM25 with equal weights)")


In [ ]:
# Evaluate all 6 retrievers
print("Starting evaluation of all retrievers...")

# Store all evaluation results
all_results = {}

# 1. Naive Retriever
naive_results = evaluate_retriever_chain(naive_retrieval_chain, "naive", golden_dataset)
all_results["naive"] = naive_results

# 2. BM25 Retriever  
bm25_results = evaluate_retriever_chain(bm25_retrieval_chain, "bm25", golden_dataset)
all_results["bm25"] = bm25_results

# 3. Contextual Compression Retriever
compression_results = evaluate_retriever_chain(contextual_compression_retrieval_chain, "compression", golden_dataset)
all_results["compression"] = compression_results

# 4. Multi-Query Retriever
multi_query_results = evaluate_retriever_chain(multi_query_retrieval_chain, "multi_query", golden_dataset)
all_results["multi_query"] = multi_query_results

# 5. Parent Document Retriever
parent_doc_results = evaluate_retriever_chain(parent_document_retrieval_chain, "parent_document", golden_dataset)
all_results["parent_document"] = parent_doc_results

# 6. Simple Ensemble Retriever
ensemble_results = evaluate_retriever_chain(simple_ensemble_retrieval_chain, "simple_ensemble", golden_dataset)
all_results["simple_ensemble"] = ensemble_results

print(f"\nCompleted evaluation of {len(all_results)} retrievers!")
print("Retrievers evaluated:", list(all_results.keys()))


In [ ]:
# Run RAGAS metrics on all retrievers
print("Running RAGAS metrics evaluation...")

# Define metrics
metrics = [faithfulness, answer_relevancy, context_recall, context_precision]

# Store metric results
metric_results = {}

for retriever_name, results in all_results.items():
    print(f"\nEvaluating {retriever_name} with RAGAS metrics...")
    
    try:
        # Convert results to DataFrame for RAGAS
        results_df = pd.DataFrame(results)
        
        # Run RAGAS evaluation
        ragas_scores = evaluate(
            dataset=results_df,
            metrics=metrics,
            llm=ragas_llm,
            embeddings=ragas_embeddings
        )
        
        # Store scores
        metric_results[retriever_name] = ragas_scores
        print(f"✅ {retriever_name} evaluation completed")
        print(f"   Scores: {ragas_scores}")
        
    except Exception as e:
        print(f"❌ Error evaluating {retriever_name}: {e}")
        metric_results[retriever_name] = None

print(f"\nRAGAS evaluation completed for {len(metric_results)} retrievers!")


In [ ]:
# Create comparison table with RAGAS metrics
print("Creating comparison table...")

# Extract scores into a comparison table
comparison_data = []

for retriever_name, scores in metric_results.items():
    if scores is not None:
        row = {
            "Retriever": retriever_name,
            "Faithfulness": scores.get("faithfulness", "N/A"),
            "Answer Relevancy": scores.get("answer_relevancy", "N/A"),
            "Context Recall": scores.get("context_recall", "N/A"),
            "Context Precision": scores.get("context_precision", "N/A")
        }
        comparison_data.append(row)

# Create DataFrame
comparison_df = pd.DataFrame(comparison_data)

# Display formatted table
print("\n📊 RAGAS Metrics Comparison:")
print("=" * 80)
print(comparison_df.to_string(index=False, float_format='%.3f'))
print("=" * 80)

# Save results
comparison_df.to_csv("retriever_comparison.csv", index=False)
print("\nComparison table saved as 'retriever_comparison.csv'")


In [ ]:
# Extract cost and latency data from LangSmith
print("Extracting cost and latency data from LangSmith...")

try:
    # Get runs from LangSmith for our project
    runs = langsmith_client.list_runs(
        project_name="Advanced_Retrieval_Session09",
        limit=1000  # Adjust based on your evaluation size
    )
    
    # Group runs by retriever type
    retriever_stats = {}
    
    for run in runs:
        retriever_name = run.metadata.get("retriever", "unknown")
        
        if retriever_name not in retriever_stats:
            retriever_stats[retriever_name] = {
                "total_cost": 0,
                "total_latency": 0,
                "run_count": 0
            }
        
        # Extract cost and latency (if available)
        if hasattr(run, 'total_cost') and run.total_cost:
            retriever_stats[retriever_name]["total_cost"] += run.total_cost
        
        if hasattr(run, 'latency') and run.latency:
            retriever_stats[retriever_name]["total_latency"] += run.latency
        
        retriever_stats[retriever_name]["run_count"] += 1
    
    # Calculate averages
    cost_latency_data = []
    for retriever_name, stats in retriever_stats.items():
        avg_latency = stats["total_latency"] / stats["run_count"] if stats["run_count"] > 0 else 0
        
        cost_latency_data.append({
            "Retriever": retriever_name,
            "Total Cost ($)": f"${stats['total_cost']:.4f}",
            "Avg Latency (s)": f"{avg_latency:.3f}",
            "Run Count": stats["run_count"]
        })
    
    # Create DataFrame
    cost_latency_df = pd.DataFrame(cost_latency_data)
    
    print("\n💰 Cost and Latency Analysis:")
    print("=" * 60)
    print(cost_latency_df.to_string(index=False))
    print("=" * 60)
    
    # Save results
    cost_latency_df.to_csv("cost_latency_analysis.csv", index=False)
    print("\nCost/latency analysis saved as 'cost_latency_analysis.csv'")
    
except Exception as e:
    print(f"Could not extract LangSmith data: {e}")
    print("This might be due to API limitations or timing. Check LangSmith dashboard manually.")


In [ ]:
# Create visualizations of the results
import matplotlib.pyplot as plt
import seaborn as sns

print("Creating visualizations...")

# Set up the plotting style
plt.style.use('default')
sns.set_palette("husl")

# Create figure with subplots
fig, axes = plt.subplots(2, 2, figsize=(15, 12))
fig.suptitle('RAGAS Metrics Comparison Across Retrievers', fontsize=16, fontweight='bold')

# Prepare data for plotting
plot_data = comparison_df.copy()
metric_columns = ['Faithfulness', 'Answer Relevancy', 'Context Recall', 'Context Precision']

# Convert string values to numeric where possible
for col in metric_columns:
    plot_data[col] = pd.to_numeric(plot_data[col], errors='coerce')

# Plot each metric
for idx, metric in enumerate(metric_columns):
    row = idx // 2
    col = idx % 2
    
    ax = axes[row, col]
    
    # Create bar plot
    bars = ax.bar(plot_data['Retriever'], plot_data[metric], alpha=0.8)
    
    # Customize plot
    ax.set_title(f'{metric}', fontweight='bold')
    ax.set_ylabel('Score')
    ax.set_xlabel('Retriever')
    
    # Rotate x-axis labels for better readability
    ax.tick_params(axis='x', rotation=45)
    
    # Add value labels on bars
    for bar in bars:
        height = bar.get_height()
        if not pd.isna(height):
            ax.text(bar.get_x() + bar.get_width()/2., height + 0.01,
                   f'{height:.3f}', ha='center', va='bottom', fontsize=9)
    
    # Set y-axis limits
    ax.set_ylim(0, 1.1)

plt.tight_layout()
plt.show()

# Save the plot
plt.savefig('ragas_metrics_comparison.png', dpi=300, bbox_inches='tight')
print("Visualization saved as 'ragas_metrics_comparison.png'")


## 📊 Analysis and Reflection

Based on the comprehensive evaluation of 6 different retriever methods using RAGAS metrics and LangSmith tracking, here are the key findings and recommendations:


In [ ]:
# Automated Analysis and Recommendations
print("🔍 Automated Analysis and Recommendations")
print("=" * 60)

# Calculate overall performance scores (average of all metrics)
overall_scores = {}
for retriever_name, scores in metric_results.items():
    if scores is not None:
        # Calculate average of all metrics
        metric_values = [v for v in scores.values() if isinstance(v, (int, float))]
        if metric_values:
            overall_scores[retriever_name] = sum(metric_values) / len(metric_values)

# Sort retrievers by overall performance
sorted_retrievers = sorted(overall_scores.items(), key=lambda x: x[1], reverse=True)

print("\n🏆 Overall Performance Ranking:")
for i, (retriever, score) in enumerate(sorted_retrievers, 1):
    print(f"{i}. {retriever}: {score:.3f}")

# Find best performer
best_retriever = sorted_retrievers[0][0] if sorted_retrievers else "N/A"
best_score = sorted_retrievers[0][1] if sorted_retrievers else 0

print(f"\n🥇 Best Performing Retriever: {best_retriever} (Score: {best_score:.3f})")

# Analysis insights
print("\n💡 Key Insights:")
print("- Contextual Compression typically shows high precision due to reranking")
print("- Multi-Query Retrieval often improves recall by generating diverse queries")
print("- Parent Document Retrieval balances precision with broader context")
print("- BM25 excels at exact term matching but may miss semantic relationships")
print("- Ensemble methods combine strengths but may increase latency")

print("\n⚖️ Trade-offs Summary:")
print("- Precision vs Recall: Higher precision often means lower recall")
print("- Cost vs Performance: More sophisticated retrievers cost more")
print("- Latency vs Quality: Complex retrievers take longer but may perform better")

print("\n🎯 Recommendations for Projects with Domains CSV:")
print("1. For cost-sensitive applications: Use Naive or BM25 retrievers")
print("2. For high-quality responses: Use Contextual Compression or Multi-Query")
print("3. For balanced performance: Use Parent Document or Simple Ensemble")
print("4. For production systems: Consider ensemble methods for robustness")

print("\n" + "=" * 60)
print("✅ Evaluation Complete! Check the generated CSV files and visualizations.")
